**<h1 align="center">Dataset Construction (JobHop × ESCO)</h1>**

This notebook constructs the final datasets used in the thesis experiments by integrating ESCO job information with the JobHop career trajectory dataset. First, ESCO occupations_en.csv and occupationSkillRelations_en.csv are merged to create job descriptions containing occupationUri, iscoGroup, preferredLabel, description, associated essential skills, and a consolidated job_text field (“Job Title. Job Description. Essential Skills”).

Next, the JobHop dataset is loaded from Hugging Face (already split into train/validation/test). To prevent data leakage, all preprocessing steps are applied independently within each split. These steps include removing duplicate rows and filtering unknown or invalid occupation codes and labels, grouping experiences by person_id and retaining only individuals with at least two experiences, and constructing chronological timelines by ordering experiences by start date. Consecutive duplicate occupations are removed while preserving alignment between occupations and associated skills.

For each person, the ordered occupation sequence defines the CV history (cv_codes, cv_labels). Essential ESCO skills associated with each past occupation are attached at the experience level and retained as aligned lists. During CV text construction, up to a fixed number of essential skills per past occupation are included to ensure balanced representation across career stages and to prevent earlier roles from dominating the skill list. The resulting cv_text combines work experience and selected essential skills.

The next occupation in the sequence defines the prediction target (target_code, target_label). ESCO job descriptions are then joined to the target occupation by aligning JobHop target labels with ESCO preferredLabel, handling label and version mismatches. The final output consists of three finalized splits (train/validation/test) containing: person_id, cv_codes, cv_labels, cv_text, target_code, target_label, jd_text, occupationUri, and iscoGroup, ready for negative sampling, SBERT training, and evaluation.

**ESCO dataset - v1.2.1 / classification / en / csv** available at: https://esco.ec.europa.eu/en/use-esco/download

**JobHop** dataset available at: https://huggingface.co/datasets/aida-ugent/JobHop


<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [1]:
import pandas as pd
import numpy as np
import os
import hashlib
import random

from datasets import load_dataset

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [3]:
# ESCO FILES
occ = pd.read_csv("../Data/ESCO_Files/occupations_en.csv")
occrs = pd.read_csv("../Data/ESCO_Files/occupationSkillRelations_en.csv")
skills = pd.read_csv("../Data/ESCO_Files/skills_en.csv")

In [4]:
# JOBHOP
ds = load_dataset("aida-ugent/JobHop")

train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()
test_df = test_ds.to_pandas()

<a class="anchor" id="chapter3"></a>

# 3. ESCO Initial Analysis

</a>

<a class="anchor" id="sub-section-3_1"></a>

## 3.1. Occupations

</a>

<a class="anchor" id="sub-section-3_1_1"></a>

### 3.1.1. Types & Structure

</a>

In [5]:
print("Shape of Occupations df:", occ.shape)
occ.head()

Shape of Occupations df: (3043, 15)


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,director of technical arts\ntechnical supervis...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Technical directors realise the artistic visio...,2654.1.7,http://data.europa.eu/ux2/nace2.1/9031
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,wire drawer\nforming machine operative\ndraw m...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,http://data.europa.eu/ux2/nace2.1/242
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,precision device quality control supervisor\np...,NaN,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Precision device inspectors make sure precisio...,7543.10.3,http://data.europa.eu/ux2/nace2.1/2651
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2024-01-29T16:01:13.998Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1,http://data.europa.eu/ux2/nace2.1/5223
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,yield manager\nhospitality yields manager\nhos...,NaN,released,2024-01-11T10:28:45.871Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9,"http://data.europa.eu/ux2/nace2.1/701,\nhttp:/..."


In [6]:
occ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   conceptType              3043 non-null   object
 1   conceptUri               3043 non-null   object
 2   iscoGroup                3043 non-null   int64 
 3   preferredLabel           3043 non-null   object
 4   altLabels                3043 non-null   object
 5   hiddenLabels             8 non-null      object
 6   status                   3043 non-null   object
 7   modifiedDate             3043 non-null   object
 8   regulatedProfessionNote  3043 non-null   object
 9   scopeNote                310 non-null    object
 10  definition               8 non-null      object
 11  inScheme                 3043 non-null   object
 12  description              3043 non-null   object
 13  code                     3043 non-null   object
 14  naceCode                 3043 non-null  

<a class="anchor" id="sub-section-3_1_2"></a>

### 3.1.2. Duplicates

</a>

In [7]:
occ[occ.index.duplicated() == True]

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode


In [8]:
duplicates = occ.duplicated().sum()
print(f"There are {duplicates} duplicated rows.")

There are 0 duplicated rows.


<a class="anchor" id="sub-section-3_1_3"></a>

### 3.1.3. Missing Values

</a>

In [9]:
occ.isna().sum()

conceptType                   0
conceptUri                    0
iscoGroup                     0
preferredLabel                0
altLabels                     0
hiddenLabels               3035
status                        0
modifiedDate                  0
regulatedProfessionNote       0
scopeNote                  2733
definition                 3035
inScheme                      0
description                   0
code                          0
naceCode                      0
dtype: int64

<a class="anchor" id="sub-section-3_1_4"></a>

### 3.1.4. Statistical Analysis

</a>

In [10]:
occ.describe(include = object).T

,count,unique,top,freq
conceptType,3043,1,Occupation,3043
conceptUri,3043,3039,http://data.europa.eu/esco/occupation/4d27152a...,2
preferredLabel,3043,3039,early years teaching assistant,2
altLabels,3043,3039,reception teaching assistant\nassistant in ear...,2
hiddenLabels,8,8,sexual health consultant\nyouth policy manager...,1
status,3043,1,released,3043
modifiedDate,3043,2902,2025-06-25T07:45:00.959Z,27
regulatedProfessionNote,3043,2,http://data.europa.eu/esco/regulated-professio...,3027
scopeNote,310,274,Excludes people performing managerial activities.,12
definition,8,8,Excludes choreologist.,1


<a class="anchor" id="sub-section-3_1_5"></a>

### 3.1.5. Drop Columns

</a>

To construct the job descriptions, I will only need:
- conceptUri: ESCO occupation ID
- isoGroup: ESCO code
- preferredLabel: Job title
- description: Job description text

In [11]:
print(list(occ))

['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code', 'naceCode']


In [12]:
occ = occ.drop(['conceptType', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 
                'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'code', 
                'naceCode'], axis=1)

In [13]:
occ = occ.rename(columns={"conceptUri": "occupationUri"})

In [14]:
occ.head()

,occupationUri,iscoGroup,preferredLabel,description
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,Metal drawing machine operators set up and ope...
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,Precision device inspectors make sure precisio...
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,Air traffic safety technicians provide technic...
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,Hospitality revenue managers maximise revenue ...


<a class="anchor" id="sub-section-3_2"></a>

## 3.2. Occupation Skill Relations

</a>

<a class="anchor" id="sub-section-3_2_1"></a>

### 3.2.1. Types & Structure

</a>

In [15]:
print("Shape of Occupation Skill Relations df:", occrs.shape)
occrs.head()

Shape of Occupation Skill Relations df: (126051, 6)


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands


In [16]:
occrs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126051 entries, 0 to 126050
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   occupationUri    126051 non-null  object
 1   occupationLabel  126051 non-null  object
 2   relationType     126051 non-null  object
 3   skillType        125992 non-null  object
 4   skillUri         126051 non-null  object
 5   skillLabel       126051 non-null  object
dtypes: object(6)
memory usage: 5.8+ MB


<a class="anchor" id="sub-section-3_2_2"></a>

### 3.2.2. Duplicates

</a>

In [17]:
occrs[occrs.index.duplicated() == True]

,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel


In [18]:
duplicates = occrs.duplicated().sum()
print(f"There are {duplicates} duplicated rows.")

There are 0 duplicated rows.


<a class="anchor" id="sub-section-3_2_3"></a>

### 3.2.3. Missing Values

</a>

In [19]:
occrs.isna().sum()

occupationUri       0
occupationLabel     0
relationType        0
skillType          59
skillUri            0
skillLabel          0
dtype: int64

<a class="anchor" id="sub-section-3_2_4"></a>

### 3.2.4. Statistical Analysis

</a>

In [20]:
occrs.describe(include = object).T

,count,unique,top,freq
occupationUri,126051,3039,http://data.europa.eu/esco/occupation/9b889f07...,178
occupationLabel,126051,3039,specialised doctor,178
relationType,126051,2,essential,67600
skillType,125992,2,skill/competence,91608
skillUri,126051,13475,http://data.europa.eu/esco/skill/415abd43-e8e5...,340
skillLabel,126051,13475,use different communication channels,340


In [21]:
occrs['relationType'].unique()

array(['essential', 'optional'], dtype=object)

In [22]:
# Filter to only essential skills
essential_skills = occrs[occrs['relationType'] == 'essential']

In [23]:
essential_skills['relationType'].describe()

count         67600
unique            1
top       essential
freq          67600
Name: relationType, dtype: object

In [24]:
# Group by skill labels by occupationUri
occ_to_skills = (
    essential_skills
    .groupby('occupationUri')['skillLabel']
    .apply(list)
    .reset_index(name='essential_skills')
)

In [25]:
print(occ_to_skills.shape)
occ_to_skills.head()

(3039, 2)


,occupationUri,essential_skills
0,http://data.europa.eu/esco/occupation/00030d09...,"[theatre techniques, organise rehearsals, writ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,"[quality and cycle time optimisation, types of..."
2,http://data.europa.eu/esco/occupation/0019b951...,"[quality assurance procedures, precision engin..."
3,http://data.europa.eu/esco/occupation/0022f466...,"[aircraft flight control systems, safety engin..."
4,http://data.europa.eu/esco/occupation/002da35b...,"[property management software, produce statist..."


In [26]:
occ_to_skills['essential_skills'].apply(len).describe()

count    3039.000000
mean       22.244159
std        11.991231
min         3.000000
25%        14.000000
50%        19.000000
75%        27.000000
max        99.000000
Name: essential_skills, dtype: float64

There are 3043 occupations in the Occupations dataframe, while in this dataframe there are only 3039. 4 occupations with no essential skills.

<a class="anchor" id="sub-section-3_3"></a>

## 3.3. Merge Occupations and Occupation Skill Relations

</a>

In [27]:
jd_df = (
    occ
    .merge(
        occ_to_skills,
        on='occupationUri',
        how='left'
    )
)

In [28]:
print("Shape: ", jd_df.shape)
jd_df.head()

Shape:  (3043, 5)


,occupationUri,iscoGroup,preferredLabel,description,essential_skills
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,Technical directors realise the artistic visio...,"[theatre techniques, organise rehearsals, writ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,Metal drawing machine operators set up and ope...,"[quality and cycle time optimisation, types of..."
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,Precision device inspectors make sure precisio...,"[quality assurance procedures, precision engin..."
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,Air traffic safety technicians provide technic...,"[aircraft flight control systems, safety engin..."
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,Hospitality revenue managers maximise revenue ...,"[property management software, produce statist..."


In [29]:
jd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   occupationUri     3043 non-null   object
 1   iscoGroup         3043 non-null   int64 
 2   preferredLabel    3043 non-null   object
 3   description       3043 non-null   object
 4   essential_skills  3043 non-null   object
dtypes: int64(1), object(4)
memory usage: 119.0+ KB


In [30]:
jd_df[jd_df.index.duplicated() == True]

# Cannot check duplicate rows because essential_skills contains a list

,occupationUri,iscoGroup,preferredLabel,description,essential_skills


In [31]:
jd_df['essential_skills'].apply(len).describe()

count    3043.000000
mean       22.237266
std        11.987292
min         3.000000
25%        14.000000
50%        19.000000
75%        27.000000
max        99.000000
Name: essential_skills, dtype: float64

In [32]:
# Create job descriptions
def build_jd_text(row):
    text = f"{row['preferredLabel']}. {row['description']}"
    if row['essential_skills']:
        skills = "; ".join(row['essential_skills'])
        text += f" Essential skills: {skills}."
    return text

Structure: **Job Title. Job Description. Essential skills: list of skills.**

In [33]:
jd_df['jd_text'] = jd_df.apply(build_jd_text, axis=1)

In [34]:
jd_df['jd_text'].head()

0    technical director. Technical directors realis...
1    metal drawing machine operator. Metal drawing ...
2    precision device inspector. Precision device i...
3    air traffic safety technician. Air traffic saf...
4    hospitality revenue manager. Hospitality reven...
Name: jd_text, dtype: object

<a class="anchor" id="chapter4"></a>

# 4. JobHop Initial Analysis

</a>

<a class="anchor" id="sub-section-4_1"></a>

## 4.1. Types & Structure

</a>

In [35]:
print("Shape of train df:", train_df.shape)
train_df.head(10)

Shape of train df: (1341832, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True
5,0,move manager,Move managers coordinate all the resources and...,1324.4,Q1 2006,Q3 2008,True
6,0,customer contact centre information clerk,Customer contact centre information clerks pro...,4222.1,Q1 2004,Q4 2007,True
7,0,ICT help desk manager,ICT help desk managers monitor the delivery of...,3512.2,Q1 2002,Q1 2004,True
8,0,ICT help desk agent,ICT help desk agents provide technical assista...,3512.1,Q1 2000,Q1 2002,True
9,0,language school teacher,Language school teachers educate non-age-speci...,2353.1,Q3 1996,Q1 2000,True


In [36]:
print("Shape of validation df:", val_df.shape)
val_df.head()

Shape of validation df: (167945, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,5,medical administrative assistant,Medical administrative assistants work very cl...,3344.1,Q1 2013,Q4 2013,True
1,5,administrative assistant,Administrative assistants provide administrati...,3343.1,Q4 2013,Q4 2018,True
2,5,room attendant,"Room attendants clean, tidy and restock guest ...",9112.4,None,None,True
3,6,unknown,unknown,None,Q1 2010,Q4 2011,True
4,6,unknown,unknown,None,Q1 2009,None,True


In [37]:
print("Shape of test df:", test_df.shape)
test_df.head()

Shape of test df: (167924, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,16,psychology lecturer,"Psychology lecturers are subject professors, t...",2310.1.35,Q1 2012,Q2 2012,True
1,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q1 2009,Q3 2011,True
2,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q4 2003,Q4 2008,True
3,16,human resources manager,"Human resources managers plan, design and impl...",1212.2,Q3 2000,Q4 2003,True
4,36,leaflet distributor,"Leaflet distributors hand out flyers, leaflet ...",9510.1,Q1 2005,Q4 2006,False


In [38]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341832 entries, 0 to 1341831
Data columns (total 7 columns):
 #   Column               Non-Null Count    Dtype 
---  ------               --------------    ----- 
 0   person_id            1341832 non-null  int64 
 1   matched_label        1341832 non-null  object
 2   matched_description  1341832 non-null  object
 3   matched_code         1330799 non-null  object
 4   start_date           1187091 non-null  object
 5   end_date             1043416 non-null  object
 6   university_studies   1341832 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 62.7+ MB


In [39]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167945 entries, 0 to 167944
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167945 non-null  int64 
 1   matched_label        167945 non-null  object
 2   matched_description  167945 non-null  object
 3   matched_code         166430 non-null  object
 4   start_date           148683 non-null  object
 5   end_date             130359 non-null  object
 6   university_studies   167945 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


In [40]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167924 entries, 0 to 167923
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167924 non-null  int64 
 1   matched_label        167924 non-null  object
 2   matched_description  167924 non-null  object
 3   matched_code         166596 non-null  object
 4   start_date           149029 non-null  object
 5   end_date             131111 non-null  object
 6   university_studies   167924 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


<a class="anchor" id="sub-section-4_2"></a>

## 4.2. Duplicates

</a>

In [41]:
print(train_df[train_df.index.duplicated() == True])
print(val_df[val_df.index.duplicated() == True])
print(test_df[test_df.index.duplicated() == True])

Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []


In [42]:
print("Train: ", train_df.duplicated().mean())
print("Validation: ", val_df.duplicated().mean())
print("Test: ", test_df.duplicated().mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [43]:
train_duplicates = train_df.duplicated().sum()
val_duplicates = val_df.duplicated().sum()
test_duplicates = test_df.duplicated().sum()
print(f"There are {train_duplicates} duplicated rows in train_df.")
print(f"There are {val_duplicates} duplicated rows in val_df.")
print(f"There are {test_duplicates} duplicated rows in test_df.")

There are 35340 duplicated rows in train_df.
There are 4463 duplicated rows in val_df.
There are 4307 duplicated rows in test_df.


In [44]:
print("Train: ", train_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Validation: ", val_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Test: ", test_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [45]:
train_df = train_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
val_df = val_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
test_df = test_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])

<a class="anchor" id="sub-section-4_3"></a>

## 4.3. Missing Values

</a>

In [46]:
train_df.isna().sum()

person_id                   0
matched_label               0
matched_description         0
matched_code            10223
start_date             134229
end_date               274907
university_studies          0
dtype: int64

In [47]:
val_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1377
start_date             16643
end_date               34551
university_studies         0
dtype: int64

In [48]:
test_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1250
start_date             16422
end_date               33955
university_studies         0
dtype: int64

<a class="anchor" id="sub-section-4_4"></a>

## 4.4. Statistical Analysis

</a>

In [49]:
train_df.describe(include = object).T

,count,unique,top,freq
matched_label,1306492,2955,unknown,210964
matched_description,1306492,2955,unknown,210964
matched_code,1296269,2955,unknown,200741
start_date,1172263,227,Q1 2011,61816
end_date,1031585,216,Q4 2012,59030


In [50]:
val_df.describe(include = object).T

,count,unique,top,freq
matched_label,163482,2606,unknown,26531
matched_description,163482,2606,unknown,26531
matched_code,162105,2606,unknown,25154
start_date,146839,203,Q1 2011,7711
end_date,128931,190,Q4 2012,7381


In [51]:
test_df.describe(include = object).T

,count,unique,top,freq
matched_label,163617,2619,unknown,26432
matched_description,163617,2619,unknown,26432
matched_code,162367,2619,unknown,25182
start_date,147195,203,Q1 2011,7639
end_date,129662,194,Q4 2013,7312


In [52]:
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.15364885510205956
Unknown rate for validation:  0.15386403396092535
Unknown rate for test:  0.15390821247180916


In [53]:
print("Unknown rate for train: ", (train_df['matched_label'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_label'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_label'] == 'unknown').mean())

Unknown rate for train:  0.16147362555606923
Unknown rate for validation:  0.16228697960631752
Unknown rate for test:  0.16154800540286157


These rows do not have a valid ESCO occupation mapping. The original resume text could not be aligned to any ESCO occupation, therefore: no occupation, no ESCO skills, no ontology relations. These rows are structurally unusable for ESCO-augmented CV–JD matching.

In [54]:
def remove_unknowns(df):
    return df[
        (df["matched_code"] != "unknown") &
        (df["matched_label"] != "unknown")
    ].copy()

In [55]:
train_df = remove_unknowns(train_df)
val_df = remove_unknowns(val_df)
test_df = remove_unknowns(test_df)

In [56]:
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.0
Unknown rate for validation:  0.0
Unknown rate for test:  0.0


In [57]:
print("Unknown rate for train: ", (train_df['matched_label'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_label'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_label'] == 'unknown').mean())

Unknown rate for train:  0.0
Unknown rate for validation:  0.0
Unknown rate for test:  0.0


In [58]:
print("Train shape: ", train_df.shape)
train_df.describe(include = object).T

Train shape:  (1095528, 7)


,count,unique,top,freq
matched_label,1095528,2954,administrative assistant,56538
matched_description,1095528,2954,Administrative assistants provide administrati...,56538
matched_code,1095528,2954,3343.1,56538
start_date,962862,224,Q1 2011,50775
end_date,824954,212,Q4 2012,46531


<a class="anchor" id="chapter5"></a>

# 5. Creating Dataset

</a>

<a class="anchor" id="sub-section-5_1"></a>

## 5.1. Logic Check

</a>

**Check people with less than 2 experiences**

In [59]:
train_df.groupby('person_id')['matched_code'].nunique().describe()

count    276692.000000
mean          3.330273
std           1.910435
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          18.000000
Name: matched_code, dtype: float64

In [60]:
print("People with ≥ 2 occupations in train: ", (train_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in val: ", (val_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in test: ", (test_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())

People with ≥ 2 occupations in train:  0.830381796365634
People with ≥ 2 occupations in val:  0.8303811367642384
People with ≥ 2 occupations in test:  0.8304957363780893


For modeling and evaluation, the sample should be restricted to individuals with at least two distinct occupations, as career transitions are required to construct CV–job pairs. Individuals with a single recorded occupation were retained for descriptive analysis but excluded from supervised matching experiments.

In [61]:
def valid_persons(df):
    return df.groupby('person_id')['matched_code'].nunique().loc[lambda x: x >= 2].index

train_df = train_df[train_df['person_id'].isin(valid_persons(train_df))]
val_df = val_df[val_df['person_id'].isin(valid_persons(val_df))]
test_df = test_df[test_df['person_id'].isin(valid_persons(test_df))]

In [62]:
#train_df.groupby('person_id')['matched_code'].nunique().describe()
#val_df.groupby('person_id')['matched_code'].nunique().describe()
test_df.groupby('person_id')['matched_code'].nunique().describe()

count    28731.000000
mean         3.814869
std          1.759112
min          2.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         16.000000
Name: matched_code, dtype: float64

**Check date consistency**

In [63]:
def quarter_to_timestamp(q):
    if pd.isna(q):
        return np.nan
    try:
        quarter, year = q.split()
        year = int(year)
        q_num = int(quarter[1])
        month = (q_num - 1) * 3 + 1
        return pd.Timestamp(year=year, month=month, day=1)
    except:
        return np.nan

In [64]:
train_df['start_ts'] = train_df['start_date'].apply(quarter_to_timestamp)
train_df['end_ts'] = train_df['end_date'].apply(quarter_to_timestamp)

val_df['start_ts'] = val_df['start_date'].apply(quarter_to_timestamp)
val_df['end_ts'] = val_df['end_date'].apply(quarter_to_timestamp)

test_df['start_ts'] = test_df['start_date'].apply(quarter_to_timestamp)
test_df['end_ts'] = test_df['end_date'].apply(quarter_to_timestamp)

In [65]:
train_df[['start_date', 'end_date', 'start_ts', 'end_ts']].sample(10)

,start_date,end_date,start_ts,end_ts
182878,Q1 2008,Q4 2009,2008-01-01,2009-10-01
270229,Q1 2015,None,2015-01-01,NaT
1215220,Q1 2012,Q4 2013,2012-01-01,2013-10-01
1048977,Q1 1997,Q4 1998,1997-01-01,1998-10-01
670773,Q3 2015,Q3 2016,2015-07-01,2016-07-01
658358,None,None,NaT,NaT
1223470,Q1 2014,None,2014-01-01,NaT
371749,Q1 2006,None,2006-01-01,NaT
1198627,Q1 2007,Q4 2007,2007-01-01,2007-10-01
626382,Q1 2012,None,2012-01-01,NaT


In [66]:
print("Invalid rate train: ", (train_df['start_ts'] > train_df['end_ts']).mean())
print("Invalid rate val: ", (val_df['start_ts'] > val_df['end_ts']).mean())
print("Invalid rate test: ", (test_df['start_ts'] > test_df['end_ts']).mean())

Invalid rate train:  1.6440353912465752e-05
Invalid rate val:  1.5473289234459014e-05
Invalid rate test:  1.5443896186129837e-05


Quarter-level dates are converted to timestamps for temporal ordering. Temporal inconsistencies affect fewer than 0.01% of records across all splits and can be tolerated. Career sequences should be ordered by start date, with end date used as a fallback.

**Add ESCO Skills**

In [67]:
occ_to_skills_cv = (
    essential_skills
    .groupby('occupationLabel')['skillLabel']
    .apply(list)
    .reset_index(name='essential_skills')
)

In [68]:
occ_to_skills_cv["matched_label"] = occ_to_skills_cv["occupationLabel"].str.lower().str.strip()
occ_to_skills_cv.head()

,occupationLabel,essential_skills,matched_label
0,3D animator,"[particle animation, principles of animation, ...",3d animator
1,3D modeller,"[3D texturing, 3D lighting, augmented reality,...",3d modeller
2,3D printing technician,"[3D printing process, printing on large scale ...",3d printing technician
3,ATM repair technician,"[mechanical tools, security threats, ATM syste...",atm repair technician
4,EU funds manager,"[urban planning law, project management princi...",eu funds manager


In [69]:
train_df["matched_label"] = train_df["matched_label"].str.lower().str.strip()
val_df["matched_label"] = val_df["matched_label"].str.lower().str.strip()
test_df["matched_label"] = test_df["matched_label"].str.lower().str.strip()

In [70]:
train_df = train_df.merge(occ_to_skills_cv, on='matched_label', how='left')
val_df = val_df.merge(occ_to_skills_cv, on='matched_label', how='left')
test_df = test_df.merge(occ_to_skills_cv, on='matched_label', how='left')

In [71]:
train_df.head()

,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies,start_ts,end_ts,occupationLabel,essential_skills
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True,2016-01-01,2019-04-01,resource manager,"[corporate social responsibility, project mana..."
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True,2017-01-01,2019-04-01,health and safety officer,"[health, safety and hygiene legislation, techn..."
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True,2013-01-01,2016-01-01,integration engineer,"[procurement of ICT network equipment, inter-o..."
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True,2012-04-01,2013-01-01,programme manager,"[program management, corporate social responsi..."
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True,2011-01-01,2012-04-01,product development engineering drafter,"[engineering principles, circular economy, des..."


<a class="anchor" id="sub-section-5_2"></a>

## 5.2. Constructing CVs

</a>

In [72]:
def dedup_consecutive_aligned(codes, labels, skills):
    out_codes, out_labels, out_skills = [], [], []
    prev_code = object()

    for c, l, s in zip(codes, labels, skills):
        if c != prev_code:
            out_codes.append(c)
            out_labels.append(l)
            out_skills.append(s)
        prev_code = c

    return out_codes, out_labels, out_skills

In [73]:
def build_timelines(df, remove_consecutive_duplicates=True):
    df = df.copy()

    # Ensure skills is always a list
    df["essential_skills"] = df["essential_skills"].apply(lambda x: x if isinstance(x, list) else [])

    # Sort chronologically within person
    df = df.sort_values(
        by=["person_id", "start_ts", "end_ts"],
        ascending=[True, True, True],
        na_position="last",
    )

    # Aggregate into aligned sequences
    timelines = (
        df.groupby("person_id")
          .apply(lambda g: pd.Series({
              "occupation_codes": g["matched_code"].tolist(),
              "occupation_labels": g["matched_label"].tolist(),
              "essential_skills": g["essential_skills"].tolist(),  # list-of-lists (aligned to experiences)
          }))
          .reset_index()
    )

    # Remove consecutive duplicates
    if remove_consecutive_duplicates:
        timelines[["occupation_codes", "occupation_labels", "essential_skills"]] = timelines.apply(
            lambda r: pd.Series(dedup_consecutive_aligned(
                r["occupation_codes"], r["occupation_labels"], r["essential_skills"]
            )),
            axis=1
        )

    return timelines

In [74]:
train_timelines = build_timelines(train_df)
val_timelines = build_timelines(val_df)
test_timelines = build_timelines(test_df)

/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_1574/793621034.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_1574/793621034.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_1574/793621034.py:17: FutureWarning: DataFrameGro

In [75]:
assert (
    (train_timelines["occupation_codes"].apply(len)
     == train_timelines["occupation_labels"].apply(len))
    &
    (train_timelines["occupation_codes"].apply(len)
     == train_timelines["essential_skills"].apply(len))
).all()

train_timelines["occupation_codes"].apply(len).describe()

count    229760.000000
mean          4.101227
std           2.027820
min           2.000000
25%           3.000000
50%           4.000000
75%           5.000000
max          22.000000
Name: occupation_codes, dtype: float64

In [76]:
train_timelines.sample(5)[['person_id', 'occupation_codes', 'occupation_labels', 'essential_skills']]

,person_id,occupation_codes,occupation_labels,essential_skills
121430,235324,"[1420.3, 1324.3.1, 1420.6, 2434.1, 1420.3, 243...","[sales account manager, distribution manager, ...","[[relationship marketing, characteristics of p..."
6714,12971,"[5223.7.9, 5223.4]","[computer and accessories specialised seller, ...","[[sales argumentation, characteristics of prod..."
95962,185742,"[1219.6, 4226.1, 3343.4, 3343.1]","[project manager, receptionist, management ass...","[[internal risk management policy, project man..."
196586,380955,"[3255.2, 2352.1.5, 2166.14, 5312.3, 2656.3]","[physiotherapy assistant, special educational ...",[[contribute to quality physiotherapy services...
218957,423811,"[5223.4, 3351.1, 5223.4]","[sales assistant, customs officer, sales assis...","[[characteristics of products, product compreh..."


<a class="anchor" id="sub-section-4_3"></a>

## 5.3. Constructing CV-Target Occupation pairs

</a>

In [77]:
def _flatten_dedup(list_of_lists):
    """Flatten list-of-lists and deduplicate while preserving order"""
    seen = set()
    out = []
    for lst in list_of_lists:
        if not isinstance(lst, list):
            continue
        for x in lst:
            if x not in seen:
                seen.add(x)
                out.append(x)
    return out

In [78]:
def history_to_cv_text(labels, skills_lol=None, max_roles=None, max_skills=None, skills_per_role=4):
    """
    labels: list of occupation labels (history)
    skills_lol: list-of-lists of skills aligned to labels (history)
    max_roles: keep only last k roles (None = keep all)
    max_skills: global cap after dedup (None = keep all)
    skills_per_role: keep only first k skills per occupation (None = keep all)
    """
    if max_roles is not None:
        labels = labels[-max_roles:]
        if skills_lol is not None:
            skills_lol = skills_lol[-max_roles:]

    text = "Work experience: " + "; ".join(labels)

    if skills_lol is not None:
        # Limit skills PER ROLE first so later occupations aren't starved
        if skills_per_role is not None:
            skills_lol = [
                (lst[:skills_per_role] if isinstance(lst, list) else [])
                for lst in skills_lol
            ]

        skills = _flatten_dedup(skills_lol)

        # Global cap (often unnecessary if max_roles*skills_per_role <= 40)
        if max_skills is not None:
            skills = skills[:max_skills]

        if len(skills) > 0:
            text += ". Skills: " + "; ".join(skills)

    return text

In [ ]:
def make_positive_pairs(timelines_df, max_history_roles=None, max_history_skills=None, skills_per_role=4):
    """
    timelines_df columns:
      - person_id
      - occupation_codes (list)
      - occupation_labels (list)
      - essential_skills (list-of-lists, aligned to experiences)
    """
    rows = []

    for _, r in timelines_df.iterrows():
        pid = r["person_id"]
        codes = r["occupation_codes"]
        labels = r["occupation_labels"]
        skills_lol = r["essential_skills"]

        L = len(codes)
        if L < 2:
            continue # should not happen after filtering

        for t in range(1, L):
            cv_codes = codes[:t]
            cv_labels = labels[:t]
            cv_skills_lol = skills_lol[:t] # aligned history slice

            rows.append({
                "person_id": pid,
                "t": t,
                "cv_codes": cv_codes,
                "cv_labels": cv_labels,
                "cv_skills_lol": cv_skills_lol, # keeps step-aligned skills
                "cv_text": history_to_cv_text(
                    cv_labels,
                    skills_lol=cv_skills_lol,
                    max_roles=max_history_roles,
                    max_skills=max_history_skills,
                    skills_per_role=skills_per_role
                ),
                "target_code": codes[t],
                "target_label": labels[t],
            })

    return pd.DataFrame(rows)

In [80]:
train_pairs_pos = make_positive_pairs(train_timelines, max_history_roles=5, max_history_skills=20, skills_per_role=4)
val_pairs_pos = make_positive_pairs(val_timelines, max_history_roles=5, max_history_skills=20, skills_per_role=4)
test_pairs_pos = make_positive_pairs(test_timelines, max_history_roles=5, max_history_skills=20, skills_per_role=4)

In [81]:
len(train_pairs_pos), len(val_pairs_pos), len(test_pairs_pos)

(712538, 89135, 89214)

In [82]:
train_pairs_pos.sample(5)[["cv_text", "target_label"]]

,cv_text,target_label
120623,Work experience: sales assistant; sales suppor...,companion
281841,Work experience: animator; university teaching...,sales assistant
296195,Work experience: sales manager; corporate trai...,sales support assistant
595591,Work experience: materials handler. Skills: in...,distribution centre dispatcher
421094,Work experience: stock broker. Skills: financi...,stock trader


In [83]:
train_pairs_pos.sample(5)

,person_id,t,cv_codes,cv_labels,cv_skills_lol,cv_text,target_code,target_label
683298,426408,3,"[2330.1.17, 2330.1, 2341.1]",[religious education teacher at secondary scho...,"[[religious studies, learning difficulties, in...",Work experience: religious education teacher a...,2422.12.4,education policy officer
491108,306327,2,"[3141.2.3, 2149.17]","[biology technician, textile, leather and foot...","[[life sciences, laboratory equipment, stem ce...","Work experience: biology technician; textile, ...",2131.4.4,biophysicist
402804,251317,2,"[3343.1.2, 3313.1]","[education administrator, accounting assistant]","[[customer service, budgetary principles, offi...",Work experience: education administrator; acco...,5223.6,shop assistant
488102,304445,1,[2411.1],[accountant],"[[accounting department processes, bookkeeping...",Work experience: accountant. Skills: accountin...,2411.1.7,financial auditor
78643,48903,2,"[4412.2, 2431.6]","[postman/postwoman, client relations manager]","[[geographic areas, road traffic laws, data pr...",Work experience: postman/postwoman; client rel...,2433.1,after-sales service technician


In [84]:
print("Empty CV in train: ", (train_pairs_pos["cv_text"].str.len() == 0).mean())
print("Empty CV in val: ", (val_pairs_pos["cv_text"].str.len() == 0).mean())
print("Empty CV in test: ", (test_pairs_pos["cv_text"].str.len() == 0).mean())

Empty CV in train:  0.0
Empty CV in val:  0.0
Empty CV in test:  0.0


<a class="anchor" id="sub-section-5_4"></a>

## 5.4. Merge CV-Target Occupation with JDs

</a>

In [85]:
train_pairs_pos[["target_code", "target_label"]].head()

,target_code,target_label
0,3512.1,ict help desk agent
1,3512.2,ict help desk manager
2,4222.1,customer contact centre information clerk
3,1324.4,move manager
4,3118.3.12,product development engineering drafter


In [86]:
jd_df[["occupationUri", "iscoGroup", "preferredLabel"]].head()

,occupationUri,iscoGroup,preferredLabel
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager


In [87]:
# Checking if there are no null values
jd_df["essential_skills"].isna().sum()

np.int64(0)

JobHop provides occupation mappings aligned with an earlier ESCO v1.x release. In this work, we use a recent ESCO release to construct job descriptions and ontology features. As ESCO occupation identifiers are largely stable across versions, this does not materially affect the matching task. Minor discrepancies may arise from ESCO version differences between JobHop and the ontology release used, though occupation-level identifiers remain stable.

In [88]:
# Normalize
train_pairs_pos["target_label"] = train_pairs_pos["target_label"].str.lower().str.strip()
val_pairs_pos["target_label"] = val_pairs_pos["target_label"].str.lower().str.strip()
test_pairs_pos["target_label"] = test_pairs_pos["target_label"].str.lower().str.strip()

jd_df["preferredLabel"] = jd_df["preferredLabel"].str.lower().str.strip()

In [89]:
jd_df_unique = jd_df.drop_duplicates(subset=["preferredLabel"], keep="first")

In [90]:
def merge_with_jd(pairs_pos, jd_df_unique):
    pairs_pos = pairs_pos.copy()
    pairs_pos["target_label"] = pairs_pos["target_label"].str.lower().str.strip()

    merged = pairs_pos.merge(
        jd_df_unique[["preferredLabel", "jd_text", "occupationUri", "iscoGroup"]],
        left_on="target_label",
        right_on="preferredLabel",
        how="inner"
    )

    # ensure 1 row per transition
    merged = merged.drop_duplicates(subset=["person_id", "t", "target_label"])

    return merged

Due to label-level alignment between JobHop and ESCO, some occupations mapped to multiple ESCO definitions; in such cases, a single representative job description was retained per transition.

In [91]:
train_pairs = merge_with_jd(train_pairs_pos, jd_df_unique)
val_pairs = merge_with_jd(val_pairs_pos, jd_df_unique)
test_pairs = merge_with_jd(test_pairs_pos, jd_df_unique)

In [92]:
print("Train: ", len(train_pairs) / len(train_pairs_pos))
print("Val: ", len(val_pairs) / len(val_pairs_pos))
print("Test: ", len(test_pairs) / len(test_pairs_pos))

Train:  0.9931161566119983
Val:  0.9927525663319684
Test:  0.9935996592463067


A small fraction of transitions (<1%) could not be matched to ESCO job descriptions and were excluded.

In [93]:
train_pairs.sample(3)[["cv_text", "target_label", "jd_text"]]

,cv_text,target_label,jd_text
123090,Work experience: kitchen assistant; warehouse ...,technical sales representative,technical sales representative. Technical sale...
152105,Work experience: tourist guide; proofreader; q...,classical languages teacher secondary school,classical languages teacher secondary school. ...
187312,Work experience: tennis coach; fitness instruc...,tennis coach,tennis coach. Tennis coaches advise and guide ...


In [95]:
train_pairs.head()

,person_id,t,cv_codes,cv_labels,cv_skills_lol,cv_text,target_code,target_label,preferredLabel,jd_text,occupationUri,iscoGroup
0,0,1,[2353.1],[language school teacher],"[[assessment processes, learning difficulties,...",Work experience: language school teacher. Skil...,3512.1,ict help desk agent,ict help desk agent,ICT help desk agent. ICT help desk agents prov...,http://data.europa.eu/esco/occupation/aaeec9a7...,3512
1,0,2,"[2353.1, 3512.1]","[language school teacher, ict help desk agent]","[[assessment processes, learning difficulties,...",Work experience: language school teacher; ict ...,3512.2,ict help desk manager,ict help desk manager,ICT help desk manager. ICT help desk managers ...,http://data.europa.eu/esco/occupation/1242d99a...,3512
2,0,3,"[2353.1, 3512.1, 3512.2]","[language school teacher, ict help desk agent,...","[[assessment processes, learning difficulties,...",Work experience: language school teacher; ict ...,4222.1,customer contact centre information clerk,customer contact centre information clerk,customer contact centre information clerk. Cus...,http://data.europa.eu/esco/occupation/b7b75eb6...,4222
3,0,4,"[2353.1, 3512.1, 3512.2, 4222.1]","[language school teacher, ict help desk agent,...","[[assessment processes, learning difficulties,...",Work experience: language school teacher; ict ...,1324.4,move manager,move manager,move manager. Move managers coordinate all the...,http://data.europa.eu/esco/occupation/8f928d49...,1324
4,0,5,"[2353.1, 3512.1, 3512.2, 4222.1, 1324.4]","[language school teacher, ict help desk agent,...","[[assessment processes, learning difficulties,...",Work experience: language school teacher; ict ...,3118.3.12,product development engineering drafter,product development engineering drafter,product development engineering drafter. Produ...,http://data.europa.eu/esco/occupation/c52129d9...,3118


In [98]:
pd.set_option("display.max_colwidth", None)
train_pairs["cv_text"].head()

0                                                                                                                                                                                                                                                                                                                                                                                                 Work experience: language school teacher. Skills: assessment processes; learning difficulties; instructional strategies; language teaching methods
1                                                                                                                                                                                                                                                             Work experience: language school teacher; ict help desk agent. Skills: assessment processes; learning difficulties; instructional strategies; language teaching methods; characteristics of products; ICT system user

Each dataset instance represents a career transition, pairing an accumulated CV history with a target ESCO-aligned job description, enriched with occupation hierarchy metadata for ontology-aware learning and explainability.

| Column name            | Type               | Description                                                                                                                                                                                                                |
| ---------------------- | ------------------ | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **person_id**          | Integer / String   | Unique anonymized identifier of an individual. A person may appear in multiple rows, each corresponding to a different observed career transition.                                                                         |
| **t**                  | Integer            | Temporal index of the career transition for a given individual. Indicates the position in the ordered career sequence (e.g., first transition, second transition).                                                         |
| **cv_codes**           | List[String]       | Ordered list of occupation codes representing the individual’s work history up to time *t*. Codes correspond to ISCO/ESCO-aligned occupation identifiers.                                                                  |
| **cv_labels**          | List[String]       | Human-readable occupation labels corresponding to `cv_codes`. Preserves the same chronological order as the career history.                                                                                                |
| **cv_skills_lol**      | List[List[String]] | List of lists of essential ESCO skill labels aligned with the occupational history up to time *t*. Each inner list corresponds to the essential skills associated with a single past occupation in the career sequence.    |
| **cv_text**            | String             | Serialized textual representation of the candidate’s profile up to time *t*, constructed from work experience (`cv_labels`) and associated essential skills (`cv_skills_lol`). Used as the CV input to the language model. |
| **target_code**        | String             | Occupation code of the next job in the observed career sequence. Serves as the positive target for the matching task.                                                                                                      |
| **target_label**       | String             | Human-readable label of the target occupation corresponding to `target_code`.                                                                                                                                              |
| **preferredLabel_key** | String             | ESCO preferred occupation label from the job description dataframe, used to align JobHop occupation labels with ESCO job descriptions.                                                                            |
| **jd_text**            | String             | Textual job description associated with the target occupation, constructed from ESCO occupation descriptions and essential skills. Used as the JD input to the language model.                                             |
| **occupationUri**      | String             | ESCO URI uniquely identifying the occupation concept associated with the job description.                                                                                                                                  |
| **iscoGroup**          | String             | ISCO group code of the occupation (e.g., major or sub-major group). Used for ontology-aware negative sampling and hierarchical analysis.                                                                                   |                                |

For SBERT training keep: "person_id", "t", "cv_text", "jd_text", "target_label"

<a class="anchor" id="chapter6"></a>

# 6. Saving Dataset

</a>

In [99]:
os.getcwd()

'/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Notebooks'

In [100]:
os.chdir('/Users/chloedeschanel/Desktop/NOVA IMS/Thesis/Dataset/Processed')

In [ ]:
#train_pairs.to_csv("train.csv", index=False)
#val_pairs.to_csv("val.csv", index=False)
#test_pairs.to_csv("test.csv", index=False)

In [214]:
""""""""""

import json

meta = {
    "source_dataset": "JobHop",
    "unknown_filtered": True,
    "duplicates_dropped": "person_id + matched_code + start_date + end_date",
    "min_unique_occupations_per_person": 2,

    "timeline_construction": {
        "sort_order": ["start_ts", "end_ts"],
        "consecutive_duplicates_removed": True,
        "deduplication_key": "occupation_code",
        "min_timeline_length": 2
    },

    "cv_construction": {
        "experience_source": "matched_label (JobHop)",
        "skills_source": "ESCO occupation–skill relations",
        "skills_type": "essential",
        "skills_alignment": "per-experience (list-of-lists)",
        "skills_deduplicated": True,
        "skills_capped": True
    },

    "jd_construction": {
        "source": "ESCO",
        "merge_key": "target_label (normalized) -> preferredLabel (normalized)",
        "match_rate_after_dedup": 0.9931161566119983,
        "includes_essential_skills": True
    }
}
with open("data/processed/pairs_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

"""""""""

'"\n\nimport json\n\nmeta = {\n    "source_dataset": "JobHop",\n    "unknown_filtered": True,\n    "duplicates_dropped": "person_id + matched_code + start_date + end_date",\n    "min_unique_occupations_per_person": 2,\n\n    "timeline_construction": {\n        "sort_order": ["start_ts", "end_ts"],\n        "consecutive_duplicates_removed": True,\n        "deduplication_key": "occupation_code",\n        "min_timeline_length": 2\n    },\n\n    "cv_construction": {\n        "experience_source": "matched_label (JobHop)",\n        "skills_source": "ESCO occupation–skill relations",\n        "skills_type": "essential",\n        "skills_alignment": "per-experience (list-of-lists)",\n        "skills_deduplicated": True,\n        "skills_capped": True\n    },\n\n    "jd_construction": {\n        "source": "ESCO",\n        "merge_key": "target_label (normalized) -> preferredLabel (normalized)",\n        "match_rate_after_dedup": 0.9931161566119983,\n        "includes_essential_skills": True\n   